# 🔬 Module 09: Mechanistic Interpretability & Frontiers
### *From Black-Box Networks to Transformer Circuits and Sparse Autoencoders (Anthropic 2021–2024)*

Welcome to **Module 09 of Project Synapse**. This module explores the frontier of **Mechanistic Interpretability**: reverse-engineering neural networks from weights and activations into understandable algorithms and circuits.

---

## 📚 Key Breakthroughs Covered:
1. **Elhage et al. (2021)**: *A Mathematical Framework for Transformer Circuits* — Residual stream as a shared communication bus; $QK$ and $OV$ circuits.
2. **Olsson et al. (2022)**: *In-context Learning and Induction Heads* — The microscopic 2-layer circuit $[A][B] \dots [A] 	o [B]$ that drives in-context learning.
3. **Elhage et al. (2022)**: *Toy Models of Superposition* — Why models represent $M > D$ features almost orthogonally in lower-dimensional space.
4. **Bricken et al. (2023) / Templeton et al. (2024)**: *Towards Monosemanticity & Scaling Monosemanticity* — Using overcomplete Sparse Autoencoders (SAEs) to untangle polysemantic neurons into monosemantic concepts.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'python' / 'module09_mechanistic_interpretability'))

import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

from sparse_autoencoder_sae import SparseAutoencoder, SyntheticSuperpositionData
from circuit_induction_heads import TwoLayerToyTransformer, generate_induction_sequences, train_induction_circuit

print("PyTorch version:", torch.__version__)
print("All Module 09 dependencies loaded successfully!")

## 1. The Superposition Hypothesis: Features in 2D Space

Why are neurons polysemantic? Because the world has more features than the model has dimensions ($M \gg D$).
To minimize loss, networks compress sparse features in **superposition** by placing them at angles close to orthogonal.
Below, we generate $M=5$ ground-truth features embedded in a $D=2$ dimensional hidden space.

In [ ]:
# Generate 5 ground-truth features in 2D activation space
toy_data = SyntheticSuperpositionData(d_in=2, num_features=5, sparsity=0.75)
activations, coeffs = toy_data.generate_batch(batch_size=800)

plt.figure(figsize=(7, 7), dpi=100)
# Plot activation cloud
plt.scatter(activations[:, 0].numpy(), activations[:, 1].numpy(), s=8, alpha=0.35, color='#06b6d4', label='Activations in Superposition')

# Plot ground-truth feature vectors (Pentagon vertices)
gt = toy_data.ground_truth_vectors
for i in range(len(gt)):
    plt.arrow(0, 0, gt[i, 0], gt[i, 1], head_width=0.08, head_length=0.08, fc='#f43f5e', ec='#f43f5e', lw=2)
    plt.text(gt[i, 0] * 1.15, gt[i, 1] * 1.15, f'Feature {i+1}', color='#f43f5e', fontweight='bold', fontsize=11, ha='center', va='center')

plt.axhline(0, color='gray', linestyle=':', alpha=0.5)
plt.axvline(0, color='gray', linestyle=':', alpha=0.5)
plt.title('Superposition: 5 True Features Packed into 2D Hidden Activation Space', fontsize=12)
plt.xlabel('Activation Neuron $x_1$')
plt.ylabel('Activation Neuron $x_2$')
plt.axis('equal')
plt.grid(True, alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Sparse Autoencoder (SAE) Feature Disentanglement

The SAE projects activations into an overcomplete space ($M=8$ features) with an $L_1$ sparsity penalty:
$$\mathcal{L}_{	ext{SAE}} = rac{1}{D} \|x - \hat{x}\|_2^2 + \lambda \sum_{i=1}^M |f_i(x)|$$
Columns of the decoder $W_{	ext{dec}}$ are normalized to unit norm: $\|W_{	ext{dec}, i}\|_2 = 1$.

In [ ]:
# Train Sparse Autoencoder to untangle the polysemantic activations
sae = SparseAutoencoder(d_in=2, d_sae=8, l1_coefficient=0.035)
optimizer = torch.optim.Adam(sae.parameters(), lr=0.02)

train_x, _ = toy_data.generate_batch(2000)

print("Training SAE dictionary learning...")
for epoch in range(350):
    optimizer.zero_grad()
    _, _, loss, metrics = sae(train_x)
    loss.backward()
    optimizer.step()
    sae.normalize_decoder_columns()

print(f"Training Complete! Final R^2: {metrics['explained_variance_r2']:.2%}, L0 Active Features: {metrics['l0_active_features']:.2f}")

In [ ]:
# Visualize Learned Decoder Columns vs Ground Truth Features
dec_vectors = sae.w_dec.detach().numpy().T  # (d_sae, 2)

plt.figure(figsize=(7, 7), dpi=100)
# Plot activation points
plt.scatter(activations[:400, 0].numpy(), activations[:400, 1].numpy(), s=5, alpha=0.2, color='#64748b')

# True features
for i in range(len(gt)):
    plt.arrow(0, 0, gt[i, 0], gt[i, 1], head_width=0.06, head_length=0.06, fc='#f43f5e', ec='#f43f5e', lw=2, alpha=0.6)

# Learned SAE dictionary columns
for j in range(len(dec_vectors)):
    norm = np.linalg.norm(dec_vectors[j])
    if norm > 0.1:
        plt.arrow(0, 0, dec_vectors[j, 0], dec_vectors[j, 1], head_width=0.06, head_length=0.06, fc='#10b981', ec='#10b981', lw=2.5, linestyle='--')

plt.title('Learned SAE Monosemantic Dictionary Directions (Green) vs Ground Truth (Red)', fontsize=12)
plt.axis('equal')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## 3. The 2-Layer Transformer Induction Head Circuit

Induction heads are the engine of in-context learning:
- **Prompt**: `[..., A, B, ..., A]` $\implies$ **Target**: `B`
- **Layer 0 (Previous-Token Head)**: Copies info about $A$ into the position of $B$.
- **Layer 1 (Induction Head)**: Query at current token $A$ attends to token $B$ because $B$ holds the "preceded by $A$" key!
- **OV Circuit**: Copies token $B$'s identity directly to next-token logits.

In [ ]:
# Train a 2-Layer Attention-Only Transformer on Induction Sequences
vocab_size = 20
model = TwoLayerToyTransformer(vocab_size=vocab_size, d_model=32, num_heads=2)

print("Training 2-layer transformer induction circuit...")
train_induction_circuit(model, num_steps=250, lr=0.01)

# Test sequence: [Token 4, Token 15, Distractor 8, Distractor 2, Token 4] -> Should predict 15
test_seq = torch.tensor([[4, 15, 8, 2, 4]], dtype=torch.long)
logits, attn0, attn1 = model(test_seq)

pred_next = torch.argmax(logits[0, -1, :]).item()
print(f"Prompt: [A=4, B=15, 8, 2, A=4]")
print(f"Target: 15 (Token B) | Model Prediction: {pred_next}")

In [ ]:
# Visualize Attention Heatmaps for Layer 0 vs Layer 1
tokens_labels = ['A(4)', 'B(15)', 'C(8)', 'D(2)', 'A(4)']

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=100)

# Layer 0 Head 0 Attention Map
a0 = attn0[0, 0].detach().numpy()
im0 = axes[0].imshow(a0, cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Layer 0 (Previous-Token Head)\nAttends to Token at $i-1$', fontsize=12)
axes[0].set_xticks(range(5))
axes[0].set_yticks(range(5))
axes[0].set_xticklabels(tokens_labels)
axes[0].set_yticklabels(tokens_labels)
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# Layer 1 Induction Head Attention Map
# Check which head in Layer 1 has the strongest induction spike
head_attns = [attn1[0, h, 4, 1].item() for h in range(model.num_heads)]
best_head = int(np.argmax(head_attns))
a1 = attn1[0, best_head].detach().numpy()

im1 = axes[1].imshow(a1, cmap='Purples', vmin=0, vmax=1)
axes[1].set_title(f'Layer 1 Head {best_head} (Induction Head)\nAttends from Query A(4) to Target B(15)!', fontsize=12)
axes[1].set_xticks(range(5))
axes[1].set_yticks(range(5))
axes[1].set_xticklabels(tokens_labels)
axes[1].set_yticklabels(tokens_labels)
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()